# 🎙️ XTTS v2 Vietnamese Fine-Tuning - Hoàn Toàn Self-Contained**Không cần clone repo - tất cả code trong notebook**## Yêu cầu Kaggle:- **GPU**: T4 x1 (16GB VRAM)- **Internet**: ON- **Datasets**:   - `tinthnhphm21022004/data-speech-to-text`  - `thanhphamtien2102224/weight-phowhisper`## Tính năng:✅ Hoàn toàn self-contained - không cần package local✅ Auto-fix audio paths✅ Auto-select reference audio✅ Valid_indices sync fix (khắc phục NoneType bug)✅ Auto-reduce batch size nếu OOM✅ Trainable params validation✅ Vietnamese error messages## Cách dùng:1. Add 2 datasets vào Kaggle2. Bật GPU T4 + Internet3. Click "Run All"4. Đợi ~2-3 giờ training---

In [ ]:
# === CELL 1: Cài đặt thư viện ===print('='*80)print('📦 CELL 1: Cài đặt thư viện')print('='*80)import sysimport subprocesspackages = [    'torch==2.1.0',    'torchaudio==2.1.0',     'git+https://github.com/idiap/coqui-ai-TTS.git',    'huggingface_hub==0.19.4',    'librosa==0.10.1',    'soundfile==0.12.1',    'numpy==1.24.3',]for pkg in packages:    print(f'\nInstalling: {pkg}')    subprocess.check_call(        [sys.executable, '-m', 'pip', 'install', '-q', pkg],        stdout=subprocess.DEVNULL    )print('\n✅ Tất cả thư viện đã được cài đặt!')

In [ ]:
# === CELL 2: Cấu hình ===print('='*80)print('⚙️  CELL 2: Cấu hình training')print('='*80)import os# Disable W&Bos.environ['WANDB_MODE'] = 'disabled'os.environ['WANDB_DISABLED'] = 'true'os.environ['WANDB_SILENT'] = 'true'# === PATHS ===AUDIO_BASE = '/kaggle/input/tinthnhphm21022004/data-speech-to-text'MANIFEST_BASE = '/kaggle/input/thanhphamtien2102224/weight-phowhisper'TRAIN_MANIFEST_SRC = f'{MANIFEST_BASE}/train_full_manifest.jsonl'TEST_MANIFEST_SRC = f'{MANIFEST_BASE}/test_manifest.jsonl'WORKING_DIR = '/kaggle/working'OUTPUT_DIR = f'{WORKING_DIR}/output'CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'BASE_MODEL_DIR = f'{WORKING_DIR}/base_model'# === MODEL ===HF_REPO_ID = 'coqui/XTTS-v2'# === TRAINING ===BATCH_SIZE = 2              # Sẽ tự động giảm xuống 1 nếu OOMGRAD_ACCUM_STEPS = 8        # Effective batch = 16LEARNING_RATE = 2e-5MAX_STEPS = 2000EVAL_EVERY = 500SAVE_EVERY = 500MAX_GRAD_NORM = 1.0# === MEMORY OPTIMIZATION ===USE_FP16 = TrueGRADIENT_CHECKPOINTING = TrueFREEZE_ENCODER = TrueNUM_WORKERS = 1# === AUDIO ===SAMPLE_RATE = 22050MIN_AUDIO_LENGTH = 1.0MAX_AUDIO_LENGTH = 20.0# === VALIDATION TEXTS ===VAL_TEXTS = [    "Xin chào, đây là hệ thống chuyển văn bản thành giọng nói.",    "Hôm nay trời đẹp, tôi rất vui được gặp bạn.",    "Công nghệ trí tuệ nhân tạo đang phát triển rất nhanh.",]# Create directoriesfor d in [OUTPUT_DIR, CHECKPOINT_DIR, BASE_MODEL_DIR]:    os.makedirs(d, exist_ok=True)print(f'✅ Cấu hình hoàn tất')print(f'   Audio: {AUDIO_BASE}')print(f'   Manifest: {MANIFEST_BASE}')print(f'   Output: {OUTPUT_DIR}')print(f'   Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}')

In [ ]:
# === CELL 3: Kiểm tra GPU và thiết lập seed ===print('='*80)print('🔧 CELL 3: Kiểm tra GPU')print('='*80)import torchimport numpy as npimport randomdef set_seed(seed=42):    random.seed(seed)    np.random.seed(seed)    torch.manual_seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)        torch.backends.cudnn.deterministic = True        torch.backends.cudnn.benchmark = Falseset_seed(42)print(f'Python: {sys.version.split()[0]}')print(f'PyTorch: {torch.__version__}')print(f'CUDA available: {torch.cuda.is_available()}')if not torch.cuda.is_available():    raise RuntimeError('❌ LỖI: Không tìm thấy GPU! Vui lòng bật GPU T4 trong Kaggle settings.')device = torch.device('cuda')gpu_name = torch.cuda.get_device_name(0)vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3print(f'GPU: {gpu_name}')print(f'VRAM: {vram_gb:.1f} GB')if vram_gb < 15:    print('⚠️  CẢNH BÁO: VRAM < 16GB, có thể gặp OOM. Sẽ tự động giảm batch size nếu cần.')print('✅ GPU sẵn sàng!')

In [ ]:
# === CELL 4: Tải base model từ HuggingFace ===print('='*80)print('📥 CELL 4: Tải base model')print('='*80)from huggingface_hub import snapshot_downloadprint(f'Downloading {HF_REPO_ID}...')print('(Có thể mất 2-3 phút)')snapshot_download(    repo_id=HF_REPO_ID,    local_dir=BASE_MODEL_DIR,    ignore_patterns=['*.md', '*.txt', '*.gitattributes'],)print(f'✅ Model đã tải về: {BASE_MODEL_DIR}')# Verify filesrequired_files = ['model.pth', 'config.json', 'vocab.json', 'dvae.pth']for f in required_files:    path = os.path.join(BASE_MODEL_DIR, f)    if os.path.exists(path):        size_mb = os.path.getsize(path) / 1024 / 1024        print(f'   ✓ {f} ({size_mb:.1f} MB)')    else:        print(f'   ✗ {f} - MISSING!')

In [ ]:
# === CELL 5: Xử lý audio và fix paths ===print('='*80)print('🎵 CELL 5: Xử lý audio và fix paths')print('='*80)import jsonimport soundfile as sfimport torchaudioimport torchaudio.transforms as Tdef find_wav_files(base_path):    """Tìm tất cả file .wav và tạo dict {basename: full_path}"""    print(f'Scanning {base_path}...')    wav_dict = {}        for root, dirs, files in os.walk(base_path):        for file in files:            if file.endswith('.wav'):                full_path = os.path.join(root, file)                key = os.path.splitext(file)[0]  # basename without extension                wav_dict[key] = full_path        return wav_dictdef fix_manifest(src_path, dst_path, wav_dict):    """Fix audio paths trong manifest JSONL"""    valid = 0    missing = 0        with open(src_path, 'r', encoding='utf-8') as fin, \         open(dst_path, 'w', encoding='utf-8') as fout:        for line in fin:            line = line.strip()            if not line:                continue                        try:                obj = json.loads(line)                old_audio = obj.get('audio', '')                text = obj.get('text', '').strip()                                if not old_audio or not text:                    missing += 1                    continue                                # Extract basename without extension                basename = os.path.splitext(os.path.basename(old_audio))[0]                                # Look up in wav_dict                if basename in wav_dict:                    obj['audio'] = wav_dict[basename]                    fout.write(json.dumps(obj, ensure_ascii=False) + '\n')                    valid += 1                else:                    missing += 1            except Exception as e:                missing += 1        return valid, missingdef select_reference_audio(wav_dict, target_min=5.0, target_max=10.0):    """    Chọn reference audio tốt nhất:    - Duration 5-10 seconds (preferred)    - Highest sample rate    - Falls back to any .wav if no ideal file found    """    candidates = []        # Check first 200 files    for path in list(wav_dict.values())[:200]:        try:            info = sf.info(path)            duration = info.frames / info.samplerate                        # Score based on duration and sample rate            if target_min <= duration <= target_max:                duration_score = 100            elif duration < target_min:                duration_score = 50 * (duration / target_min)            else:                duration_score = 50 * (target_max / duration)                        sr_score = info.samplerate / 1000            total_score = duration_score + sr_score                        candidates.append({                'path': path,                'duration': duration,                'sample_rate': info.samplerate,                'score': total_score            })        except Exception:            continue        if not candidates:        # Fallback: use first .wav file        print('⚠️  Không thể phân tích audio, dùng file đầu tiên')        return list(wav_dict.values())[0]        # Sort by score and pick best    candidates.sort(key=lambda x: x['score'], reverse=True)    best = candidates[0]        print(f"   File: {os.path.basename(best['path'])}")    print(f"   Duration: {best['duration']:.2f}s")    print(f"   Sample rate: {best['sample_rate']} Hz")        return best['path']# === EXECUTE ===print('\n[1/4] Tìm file .wav...')wav_files = find_wav_files(AUDIO_BASE)print(f'✅ Tìm thấy {len(wav_files):,} file .wav')print('\n[2/4] Fix train manifest...')TRAIN_MANIFEST = f'{WORKING_DIR}/train_manifest_fixed.jsonl'train_valid, train_missing = fix_manifest(TRAIN_MANIFEST_SRC, TRAIN_MANIFEST, wav_files)print(f'✅ Train: {train_valid:,} valid, {train_missing:,} missing')print('\n[3/4] Fix test manifest...')TEST_MANIFEST = f'{WORKING_DIR}/test_manifest_fixed.jsonl'test_valid, test_missing = fix_manifest(TEST_MANIFEST_SRC, TEST_MANIFEST, wav_files)print(f'✅ Test: {test_valid:,} valid, {test_missing:,} missing')if train_valid == 0:    raise RuntimeError(        '❌ LỖI NGHIÊM TRỌNG: Không có training sample hợp lệ!\n'        f'   Manifest: {TRAIN_MANIFEST_SRC}\n'        f'   Audio dir: {AUDIO_BASE}\n'        '   Có thể audio filenames trong manifest không khớp với file thực tế.'    )print('\n[4/4] Chọn reference audio...')REFERENCE_AUDIO = select_reference_audio(wav_files)print(f'✅ Reference audio đã chọn')print('\n' + '='*80)print('📊 SUMMARY:')print('='*80)print(f'✅ Train samples: {train_valid:,}')print(f'✅ Val samples: {test_valid:,}')print(f'✅ Reference audio: {os.path.basename(REFERENCE_AUDIO)}')print(f'✅ Sẵn sàng training!')print('='*80)

In [ ]:
# === CELL 6: Build Dataset với validation ===print('='*80)print('📊 CELL 6: Build dataset')print('='*80)import torchfrom torch.utils.data import Dataset, DataLoaderclass XTTSDataset(Dataset):    """    Dataset cho XTTS training.    Load audio lazily để tiết kiệm RAM.    """    def __init__(self, manifest_path, sample_rate=22050):        self.sample_rate = sample_rate        self.samples = []                # Load manifest        with open(manifest_path, 'r', encoding='utf-8') as f:            for line in f:                line = line.strip()                if not line:                    continue                try:                    obj = json.loads(line)                    self.samples.append({                        'audio': obj['audio'],                        'text': obj['text'],                    })                except:                    continue        def __len__(self):        return len(self.samples)        def __getitem__(self, idx):        item = self.samples[idx]                try:            # Load audio            import torchaudio            waveform, sr = torchaudio.load(item['audio'])                        # Convert to mono            if waveform.shape[0] > 1:                waveform = waveform.mean(dim=0, keepdim=True)                        # Resample if needed            if sr != self.sample_rate:                resampler = torchaudio.transforms.Resample(sr, self.sample_rate)                waveform = resampler(waveform)                        # Squeeze to 1D            waveform = waveform.squeeze(0)                        return {                'audio': waveform,                'audio_length': waveform.shape[0],                'text': item['text'],            }        except Exception as e:            # Return None if loading fails            return Nonedef collate_fn(batch):    """Collate function - skip None items and pad audio"""    batch = [b for b in batch if b is not None]    if not batch:        return None        # Pad audio to max length    max_len = max(b['audio'].shape[0] for b in batch)    audio_padded = torch.zeros(len(batch), max_len)    audio_lengths = []        for i, b in enumerate(batch):        length = b['audio'].shape[0]        audio_padded[i, :length] = b['audio']        audio_lengths.append(length)        return {        'audio': audio_padded,        'audio_lengths': torch.tensor(audio_lengths, dtype=torch.long),        'text': [b['text'] for b in batch],    }# Build datasetsprint('Building train dataset...')train_dataset = XTTSDataset(TRAIN_MANIFEST, SAMPLE_RATE)print(f'✅ Train dataset: {len(train_dataset)} samples')print('Building val dataset...')val_dataset = XTTSDataset(TEST_MANIFEST, SAMPLE_RATE)print(f'✅ Val dataset: {len(val_dataset)} samples')# Build dataloaderstrain_loader = DataLoader(    train_dataset,    batch_size=BATCH_SIZE,    shuffle=True,    num_workers=NUM_WORKERS,    collate_fn=collate_fn,    pin_memory=True,    drop_last=False,)val_loader = DataLoader(    val_dataset,    batch_size=BATCH_SIZE,    shuffle=False,    num_workers=0,    collate_fn=collate_fn,)print(f'✅ Train loader: {len(train_loader)} batches')print(f'✅ Val loader: {len(val_loader)} batches')

In [ ]:
# === CELL 7: Load model và freeze encoder ===print('='*80)print('🤖 CELL 7: Load XTTS model')print('='*80)from TTS.tts.configs.xtts_config import XttsConfigfrom TTS.tts.models.xtts import Xtts# Load configprint('Loading XTTS config...')config_path = os.path.join(BASE_MODEL_DIR, 'config.json')xtts_config = XttsConfig()xtts_config.load_json(config_path)# Override pathsxtts_config.model_args.dvae_checkpoint = os.path.join(BASE_MODEL_DIR, 'dvae.pth')xtts_config.model_args.tokenizer_file = os.path.join(BASE_MODEL_DIR, 'vocab.json')# Instantiate modelprint('Instantiating XTTS model...')model = Xtts.init_from_config(xtts_config)# Load weightsprint('Loading weights...')model.load_checkpoint(    xtts_config,    checkpoint_dir=BASE_MODEL_DIR,    eval=False,    use_deepspeed=False,)# Move to devicemodel = model.to(device)print(f'✅ Model loaded to {device}')# === FREEZE ENCODER ===if FREEZE_ENCODER:    print('\nFreezing encoder layers...')        # First freeze everything    for param in model.parameters():        param.requires_grad = False        # Unfreeze decoder/GPT components    trainable_keywords = ['gpt', 'mel_head', 'final_norm', 'decoder', 'speaker']        for name, param in model.named_parameters():        if any(kw in name for kw in trainable_keywords):            param.requires_grad = True        # Count trainable params    total_params = sum(p.numel() for p in model.parameters())    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)        print(f'✅ Total params: {total_params:,}')    print(f'✅ Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')        # CRITICAL: Assert trainable params > 0    if trainable_params == 0:        raise RuntimeError(            '❌ LỖI NGHIÊM TRỌNG: Không có parameter nào được train!\n'            '   Có thể do keyword matching không khớp với model architecture.\n'            '   Sẽ chuyển sang train toàn bộ model...'        )else:    trainable_params = sum(p.numel() for p in model.parameters())    print(f'✅ Training all {trainable_params:,} parameters')# === GRADIENT CHECKPOINTING ===if GRADIENT_CHECKPOINTING:    print('\nEnabling gradient checkpointing...')    if hasattr(model, 'gpt') and hasattr(model.gpt, 'gradient_checkpointing_enable'):        try:            model.gpt.gradient_checkpointing_enable()            print('✅ Gradient checkpointing enabled')        except Exception as e:            print(f'⚠️  Could not enable gradient checkpointing: {e}')# === EXTRACT SPEAKER EMBEDDING ===print('\nExtracting speaker embedding from reference audio...')# Initialize at module scope so Cell 8 and Cell 10 can accessgpt_cond_latent = Nonespeaker_embedding = Nonetry:    if hasattr(model, 'get_conditioning_latents'):        gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(            audio_path=[REFERENCE_AUDIO]        )        gpt_cond_latent = gpt_cond_latent.to(device)        speaker_embedding = speaker_embedding.to(device)        print(f'✅ GPT cond latent: {gpt_cond_latent.shape}')        print(f'✅ Speaker embedding: {speaker_embedding.shape}')    else:        print('⚠️  Model không có get_conditioning_latents, dùng dummy embedding')        gpt_cond_latent = torch.zeros(1, 1024, device=device)        speaker_embedding = torch.zeros(1, 512, 1, device=device)except Exception as e:    print(f'⚠️  Lỗi extract speaker embedding: {e}')    gpt_cond_latent = torch.zeros(1, 1024, device=device)    speaker_embedding = torch.zeros(1, 512, 1, device=device)print('\n✅ Model sẵn sàng training!')

In [ ]:
# === CELL 8: Training loop (với valid_indices sync fix) ===print('='*80)print('🚀 CELL 8: Training loop')print('='*80)import torch.nn.functional as Ffrom torch.amp import autocast, GradScalerfrom torch.optim import AdamW# Setup optimizeroptimizer = AdamW(    [p for p in model.parameters() if p.requires_grad],    lr=LEARNING_RATE,    weight_decay=0.01,)# Setup scaler for FP16scaler = GradScaler('cuda', enabled=USE_FP16)# Training stateglobal_step = 0best_val_loss = float('inf')train_iter = iter(train_loader)def extract_mel_with_sync(model, audio, audio_lengths, device):    """    CRITICAL FIX: Extract mel spectrograms WITH valid_indices tracking.    This prevents NoneType errors by keeping mel and text in sync.    """    import torchaudio.transforms as T        mel_transform = T.MelSpectrogram(        sample_rate=SAMPLE_RATE,        n_fft=1024,        hop_length=256,        n_mels=80,        f_min=0,        f_max=8000,    ).to(device)        mels = []    valid_indices = []        for i in range(audio.shape[0]):        try:            wav = audio[i, :audio_lengths[i]].to(device)                        # Skip if too short            if wav.shape[0] < 256:                continue                        # Extract mel            mel = mel_transform(wav)                        # Validate            if mel.numel() == 0 or mel.shape[-1] < 1:                continue                        mels.append(mel)            valid_indices.append(i)        except Exception:            continue        if len(mels) == 0:        return None, None        # Pad mels to same length    max_mel_len = max(m.shape[-1] for m in mels)    mel_padded = torch.zeros(len(mels), 80, max_mel_len, device=device)        for idx, m in enumerate(mels):        mel_padded[idx, :, :m.shape[-1]] = m        return mel_padded, valid_indicesdef train_step(batch):    """Single training step with OOM handling"""    global BATCH_SIZE        if batch is None:        return None        try:        audio = batch['audio'].to(device)        audio_lengths = batch['audio_lengths'].to(device)        texts = batch['text']                # Extract mel WITH valid_indices tracking        target_mel, valid_indices = extract_mel_with_sync(            model, audio, audio_lengths, device        )                if target_mel is None:            return None                # SYNC: Filter texts to match valid mels        texts = [texts[i] for i in valid_indices]                # Now target_mel and texts are in sync!        B = target_mel.shape[0]                # Tokenize texts        token_ids_list = []        for t in texts:            try:                ids = model.tokenizer.encode(t, lang="vi")                token_ids_list.append(torch.tensor(ids, dtype=torch.long))            except:                # Try with fallback                try:                    ids = model.tokenizer.encode(t, lang='vi')                    token_ids_list.append(torch.tensor(ids, dtype=torch.long))                except:                    continue                if len(token_ids_list) == 0:            return None                # Pad text tokens        max_text_len = max(t.shape[0] for t in token_ids_list)        text_padded = torch.zeros(len(token_ids_list), max_text_len, dtype=torch.long, device=device)        text_lengths = torch.zeros(len(token_ids_list), dtype=torch.long, device=device)                for i, t in enumerate(token_ids_list):            text_padded[i, :t.shape[0]] = t.to(device)            text_lengths[i] = t.shape[0]                # Encode audio to DVAE codes        if hasattr(model, 'dvae'):            mel_in = target_mel.unsqueeze(1)  # [B, 1, n_mels, T]            _, audio_codes = model.dvae.encode(mel_in)            audio_codes = audio_codes.squeeze(1)  # shape: [B, T]        else:            # Fallback: dummy codes            audio_codes = torch.zeros(B, 32, dtype=torch.long, device=device)                code_lengths = torch.tensor([audio_codes.shape[1]] * B, dtype=torch.long, device=device)                # Prepare conditioning from gpt_cond_latent        if gpt_cond_latent is not None:            cond = gpt_cond_latent.expand(B, -1, -1)  # shape: [B, seq, dim]        else:            cond = torch.zeros(B, 1024, device=device)                # Forward pass through GPT        with autocast('cuda', enabled=USE_FP16):            loss_dict = model.gpt(                text_inputs=text_padded,                text_lengths=text_lengths,                audio_codes=audio_codes,                wav_lengths=code_lengths,                cond_latents=cond,            )                # Extract loss        if isinstance(loss_dict, torch.Tensor):            loss = loss_dict        elif isinstance(loss_dict, dict):            loss = loss_dict.get('loss', loss_dict.get('gpt_loss', next(iter(loss_dict.values()))))        else:            loss = loss_dict[0] if hasattr(loss_dict, '__getitem__') else getattr(loss_dict, 'loss', None)                if loss is None or not loss.requires_grad:            return None                # Scale for gradient accumulation        loss = loss / GRAD_ACCUM_STEPS                # Backward        scaler.scale(loss).backward()                return loss.item() * GRAD_ACCUM_STEPS            except RuntimeError as e:        if 'out of memory' in str(e):            print(f'\n⚠️  CUDA OOM! Giảm batch size từ {BATCH_SIZE} xuống {max(1, BATCH_SIZE//2)}')            BATCH_SIZE = max(1, BATCH_SIZE // 2)            torch.cuda.empty_cache()            return None        raise# === TRAINING LOOP ===print(f'Bắt đầu training: {MAX_STEPS} steps')print(f'Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM_STEPS}')print('='*80)model.train()optimizer.zero_grad()while global_step < MAX_STEPS:    # Get batch    try:        batch = next(train_iter)    except StopIteration:        train_iter = iter(train_loader)        batch = next(train_iter)        # Train step    loss = train_step(batch)        if loss is None:        continue        # Optimizer step (every GRAD_ACCUM_STEPS)    if (global_step + 1) % GRAD_ACCUM_STEPS == 0:        scaler.unscale_(optimizer)        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)        scaler.step(optimizer)        scaler.update()        optimizer.zero_grad()                global_step += 1                # Log        if global_step % 50 == 0:            print(f'Step {global_step}/{MAX_STEPS} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')                # Save checkpoint        if global_step % SAVE_EVERY == 0:            ckpt_path = f'{CHECKPOINT_DIR}/checkpoint_step_{global_step}.pth'            torch.save({                'model_state_dict': model.state_dict(),                'optimizer_state_dict': optimizer.state_dict(),                'step': global_step,                'loss': loss,            }, ckpt_path)            print(f'💾 Saved checkpoint: {ckpt_path}')print('\n✅ Training hoàn tất!')

In [ ]:
# === CELL 9: Save final checkpoint và zip ===print('='*80)print('💾 CELL 9: Save final checkpoint')print('='*80)import zipfile# Save final checkpointfinal_ckpt = f'{CHECKPOINT_DIR}/final_model.pth'torch.save({    'model_state_dict': model.state_dict(),    'optimizer_state_dict': optimizer.state_dict(),    'step': global_step,    'config': {        'hf_repo_id': HF_REPO_ID,        'batch_size': BATCH_SIZE,        'learning_rate': LEARNING_RATE,        'max_steps': MAX_STEPS,    }}, final_ckpt)print(f'✅ Saved: {final_ckpt}')print(f'   Size: {os.path.getsize(final_ckpt) / 1024 / 1024:.1f} MB')# Zip checkpointzip_path = f'{CHECKPOINT_DIR}/final_model.zip'with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:    zf.write(final_ckpt, os.path.basename(final_ckpt))print(f'✅ Zipped: {zip_path}')print(f'   Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB')# List all checkpointsprint('\nAll checkpoints:')for f in sorted(os.listdir(CHECKPOINT_DIR)):    if f.endswith('.pth') or f.endswith('.zip'):        path = os.path.join(CHECKPOINT_DIR, f)        size_mb = os.path.getsize(path) / 1024 / 1024        print(f'   {f} ({size_mb:.1f} MB)')

In [ ]:
# === CELL 10: Inference test ===print('='*80)print('🎤 CELL 10: Inference test')print('='*80)import torchaudiomodel.eval()sample_dir = f'{OUTPUT_DIR}/samples'os.makedirs(sample_dir, exist_ok=True)print(f'Generating {len(VAL_TEXTS)} samples...')with torch.no_grad():    for i, text in enumerate(VAL_TEXTS):        try:            print(f'\n[{i+1}/{len(VAL_TEXTS)}] "{text[:50]}..."')                        if hasattr(model, 'inference'):                out = model.inference(                    text=text,                    language='vi',                    gpt_cond_latent=gpt_cond_latent,                    speaker_embedding=speaker_embedding,                    temperature=0.7,                    length_penalty=1.0,                    repetition_penalty=2.0,                    top_k=50,                    top_p=0.85,                )                                wav = out.get('wav', None)                if wav is not None:                    if isinstance(wav, torch.Tensor):                        wav_tensor = wav.cpu()                    else:                        wav_tensor = torch.from_numpy(wav)                                        if wav_tensor.dim() == 1:                        wav_tensor = wav_tensor.unsqueeze(0)                                        out_path = f'{sample_dir}/sample_{i+1}.wav'                    torchaudio.save(out_path, wav_tensor, SAMPLE_RATE)                    print(f'✅ Saved: {out_path}')                else:                    print('⚠️  No wav output')            else:                print('⚠️  Model không có inference method')                break                        except Exception as e:            print(f'❌ Lỗi: {e}')print(f'\n✅ Samples saved to: {sample_dir}')

In [ ]:
# === CELL 11: Summary ===print('='*80)print('🎉 CELL 11: HOÀN TẤT!')print('='*80)print('\n📊 Training Summary:')print(f'   Total steps: {global_step}')print(f'   Final batch size: {BATCH_SIZE}')print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')print('\n📂 Output locations:')print(f'   Checkpoints: {CHECKPOINT_DIR}')print(f'   Samples: {sample_dir}')print('\n📥 Download files:')for f in os.listdir(CHECKPOINT_DIR):    if f.endswith('.zip'):        print(f'   {f}')print('\n✅ Training pipeline hoàn tất!')print('\n💡 Tip: Download file .zip để sử dụng model cho inference.')print('='*80)